In [1]:
# ============================================================
# Finite Volume Method (FVM)
# 2D Heat Diffusion Animation
# MP4 Generation with PyVista
# ============================================================

# ------------------------------------------------------------
# INSTALLATION
# ------------------------------------------------------------
# !pip install pyvista imageio imageio-ffmpeg

# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------
import numpy as np
import pyvista as pv

# ------------------------------------------------------------
# DOMAIN PARAMETERS
# ------------------------------------------------------------
Lx = 1.0
Ly = 1.0

Nx = 60
Ny = 60

dx = Lx / Nx
dy = Ly / Ny

# ------------------------------------------------------------
# PHYSICAL PARAMETERS
# ------------------------------------------------------------
k = 1.0

T_left = 100.0
T_right = 0.0
T_top = 50.0
T_bottom = 75.0

# ------------------------------------------------------------
# INITIAL TEMPERATURE FIELD
# ------------------------------------------------------------
T = np.ones((Ny, Nx)) * 50.0

T[:, 0] = T_left
T[:, -1] = T_right
T[0, :] = T_bottom
T[-1, :] = T_top

# ------------------------------------------------------------
# STRUCTURED GRID FOR PYVISTA
# ------------------------------------------------------------
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)

X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)

grid = pv.StructuredGrid(X, Y, Z)

grid["Temperature"] = T.flatten(order="F")

# ------------------------------------------------------------
# CREATE PLOTTER
# ------------------------------------------------------------
plotter = pv.Plotter(off_screen=True)

mesh = plotter.add_mesh(
    grid,
    scalars="Temperature",
    cmap="inferno",
    show_edges=False,
)

plotter.view_xy()
plotter.add_axes()
plotter.show_grid()

# ------------------------------------------------------------
# OPEN VIDEO FILE
# ------------------------------------------------------------
video_name = "finite_volume_heat_diffusion.mp4"

plotter.open_movie(
    video_name,
    framerate=30,
)

# ------------------------------------------------------------
# WRITE INITIAL FRAME
# ------------------------------------------------------------
plotter.write_frame()

# ------------------------------------------------------------
# SOLVER PARAMETERS
# ------------------------------------------------------------
max_iter = 5000
tolerance = 1e-6

frame_interval = 10

print("Running simulation...")

# ------------------------------------------------------------
# ITERATIVE FVM SOLVER
# ------------------------------------------------------------
for iteration in range(max_iter):

    T_old = T.copy()

    for j in range(1, Ny - 1):
        for i in range(1, Nx - 1):

            Te = T[j, i + 1]
            Tw = T[j, i - 1]
            Tn = T[j + 1, i]
            Ts = T[j - 1, i]

            ae = k * dy / dx
            aw = k * dy / dx
            an = k * dx / dy
            a_s = k * dx / dy

            ap = ae + aw + an + a_s

            T[j, i] = (
                ae * Te +
                aw * Tw +
                an * Tn +
                a_s * Ts
            ) / ap

    # Boundary conditions
    T[:, 0] = T_left
    T[:, -1] = T_right
    T[0, :] = T_bottom
    T[-1, :] = T_top

    error = np.max(np.abs(T - T_old))

    # --------------------------------------------------------
    # SAVE FRAME EVERY N ITERATIONS
    # --------------------------------------------------------
    if iteration % frame_interval == 0:

        grid["Temperature"] = T.flatten(order="F")

        plotter.add_text(
            f"Iteration: {iteration}",
            name="iteration_text",
            font_size=12,
        )

        plotter.write_frame()

        print(
            f"Iteration {iteration:5d} "
            f"Error = {error:.6e}"
        )

    if error < tolerance:

        print("\nConverged!")
        print(f"Iterations = {iteration}")
        print(f"Error = {error:.6e}")

        grid["Temperature"] = T.flatten(order="F")
        plotter.write_frame()

        break

# ------------------------------------------------------------
# CLOSE MOVIE
# ------------------------------------------------------------
plotter.close()

print("\nVideo saved as:")
print(video_name)

Running simulation...
Iteration     0 Error = 1.875000e+01
Iteration    10 Error = 1.268797e+00
Iteration    20 Error = 6.363487e-01
Iteration    30 Error = 4.241760e-01
Iteration    40 Error = 3.179667e-01
Iteration    50 Error = 2.544380e-01
Iteration    60 Error = 2.128873e-01
Iteration    70 Error = 1.840305e-01
Iteration    80 Error = 1.618133e-01
Iteration    90 Error = 1.442723e-01
Iteration   100 Error = 1.304073e-01
Iteration   110 Error = 1.187827e-01
Iteration   120 Error = 1.093097e-01
Iteration   130 Error = 1.009516e-01
Iteration   140 Error = 9.399483e-02
Iteration   150 Error = 8.771538e-02
Iteration   160 Error = 8.220594e-02
Iteration   170 Error = 7.723449e-02
Iteration   180 Error = 7.259522e-02
Iteration   190 Error = 6.842857e-02
Iteration   200 Error = 6.453151e-02
Iteration   210 Error = 6.085421e-02
Iteration   220 Error = 5.739998e-02
Iteration   230 Error = 5.412174e-02
Iteration   240 Error = 5.105223e-02
Iteration   250 Error = 4.816729e-02
Iteration   260 

In [5]:
grid.points[:,2] = 0.15*T.flatten(order="F")

contours = grid.contour(
    isosurfaces=20,
    scalars="Temperature"
)

plotter.camera.azimuth += 1

plotter.add_mesh(
    grid,
    scalars="Temperature",
    cmap="inferno",
    scalar_bar_args={
        "title": "Temperature (°C)"
    }
)

Actor (0x74b41030fa00)
  Center:                     (0.5, 0.5, 7.5)
  Pickable:                   True
  Position:                   (0.0, 0.0, 0.0)
  Scale:                      (1.0, 1.0, 1.0)
  Visible:                    True
  X Bounds                    0.000E+00, 1.000E+00
  Y Bounds                    0.000E+00, 1.000E+00
  Z Bounds                    0.000E+00, 1.500E+01
  User matrix:                Identity
  Has mapper:                 True

Property (0x74b41030d1e0)
  Ambient:                     0.0
  Ambient color:               Color(name='light_blue', hex='#add8e6ff', opacity=255)
  Anisotropy:                  0.0
  Color:                       Color(name='light_blue', hex='#add8e6ff', opacity=255)
  Culling:                     "none"
  Diffuse:                     1.0
  Diffuse color:               Color(name='light_blue', hex='#add8e6ff', opacity=255)
  Edge color:                  Color(name='black', hex='#000000ff', opacity=255)
  Edge opacity:                1.